# Notebook 05 — ML forecasting: first supervised models

This notebook is the **first supervised ML benchmark** under the frozen Notebook 03 forecasting contract.
It is **not** a final model selection. Notebook maturation: **Phase 3 (minimal executable build)**.

Main question: **do simple, leakage-safe supervised models beat the temporal baselines** from Notebook 04 (especially persistence) under the same contract and metrics?

Primary horizon: **h24**. Sanity/contrast horizon: **h1**.


## 0. Imports, setup, and dependency preflight

Analytical question: *(none; setup only)*

Why this matters: this notebook reads/writes parquet artifacts; if parquet support is missing, execution should fail early and clearly.


In [1]:
from __future__ import annotations

import json
import itertools
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [2]:
# Dependency preflight: parquet support (read/write)
try:
    import pyarrow  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "Missing optional dependency 'pyarrow' required for parquet IO in this notebook. "
        "Install with: pip install pyarrow"
    ) from exc


In [3]:
# Reproducibility note: models here are deterministic given fixed data and fixed random_state where applicable.
RANDOM_STATE = 42
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)
RANDOM_STATE


42

## 1. Scope, frozen-contract constraints, and evaluation rules

Analytical questions:
- Under the frozen contract, does any bounded ML candidate beat the Notebook 04 baseline-to-beat on **h24 validation, slice=all**?
- If performance improves, what trade-offs change (especially **retained rows / availability** and **high-pollution bias**)?

Why this matters for the project:
- Notebook 04 established a leakage-safe baseline floor. This notebook tests whether simple supervised learning adds enough value to justify moving toward finalist modeling.

Constraints (fixed):
- Reuse **Notebook 03** label indexes and split definition. Do **not** rebuild labels or alter the split contract.
- Compare against **Notebook 04** `baseline_to_beat.parquet` (reference floor).
- No split changes, no target changes, no imputation, no deep learning, no station-specific models.
- All learned preprocessing (encoders/scalers) fits on **train only**.
- Primary horizon: **h24**. Secondary horizon: **h1** (sanity/contrast).

Interpretation boundaries:
- Treat results as **candidate evidence**, not final project conclusions.
- Select candidates using validation only; report test after selection. Do not tune on test.


## 2. Load contract, labels, baseline artifacts, and assembled history

Analytical question: are we using the correct frozen contract + baseline floor artifacts?

Why this matters: downstream comparisons only make sense if we reuse exactly the prior contract and baseline artifacts.

Required previews (in this section):
- contract summary: horizons, split boundaries, purge gap, high-pollution thresholds
- baseline-to-beat preview
- label counts by horizon × split
- assembled history shape and station count


In [4]:
def find_project_root(start_path: Path) -> Path:
    """Find the project root by walking upward from a start path.

    Markers searched (in priority order):
    - pyproject.toml
    - README.md
    - data/ directory

    The notebook must not assume the execution CWD is already the project root.
    """

    best_candidate = None
    best_score = -1

    for candidate in [start_path, *start_path.parents]:
        has_pyproject = (candidate / 'pyproject.toml').exists()
        has_readme = (candidate / 'README.md').exists()
        has_data_dir = (candidate / 'data').is_dir()

        score = 0
        score += 4 if has_pyproject else 0
        score += 2 if has_readme else 0
        score += 1 if has_data_dir else 0

        if score > best_score:
            best_candidate = candidate
            best_score = score

        if has_data_dir and (has_pyproject or has_readme):
            return candidate

    if best_candidate is not None and (best_candidate / 'data').is_dir():
        return best_candidate

    raise FileNotFoundError(
        "Could not locate PROJECT_ROOT by walking upward from the current working directory. "
        "Expected to find at least a 'data/' directory and ideally 'README.md' or 'pyproject.toml'."
    )


In [5]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# Inputs (frozen artifacts)
ASSEMBLED_PATH = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'beijing_multisite_assembled.parquet'
CONTRACT_DIR = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'forecasting_contract'
BASELINE_DIR = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'temporal_baselines'

CONTRACT_JSON_PATH = CONTRACT_DIR / 'forecasting_problem_contract.json'
SPLIT_DEF_PATH = CONTRACT_DIR / 'split_definition.parquet'
LABEL_H1_PATH = CONTRACT_DIR / 'label_index_h1.parquet'
LABEL_H24_PATH = CONTRACT_DIR / 'label_index_h24.parquet'

BASELINE_TO_BEAT_PATH = BASELINE_DIR / 'baseline_to_beat.parquet'
BASELINE_SCORE_OVERALL_PATH = BASELINE_DIR / 'baseline_scores_overall.parquet'
BASELINE_SCORE_BY_STATION_PATH = BASELINE_DIR / 'baseline_scores_by_station.parquet'
BASELINE_SCORE_HIGH_PATH = BASELINE_DIR / 'baseline_scores_high_pollution.parquet'

# Outputs (this notebook)
OUT_DIR = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'ml_first_models'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = PROJECT_ROOT / 'reports' / 'figures' / 'ml_first_models'
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT_PRED_H24 = OUT_DIR / 'ml_predictions_h24.parquet'
OUT_PRED_H1 = OUT_DIR / 'ml_predictions_h1.parquet'
OUT_SCORE_OVERALL = OUT_DIR / 'ml_scores_overall.parquet'
OUT_SCORE_BY_STATION = OUT_DIR / 'ml_scores_by_station.parquet'
OUT_SCORE_HIGH = OUT_DIR / 'ml_scores_high_pollution.parquet'
OUT_AVAILABILITY = OUT_DIR / 'availability_report.parquet'
OUT_ML_VS_BASELINE = OUT_DIR / 'ml_vs_baseline.parquet'
OUT_RUN_SUMMARY = OUT_DIR / 'ml_run_summary.json'

FIG_ML_VS_BASELINE_H24_VAL_MAE = FIG_DIR / 'ml_vs_baseline_h24_val_mae.png'
FIG_ML_FEATURE_STAGE_H24_VAL_MAE = FIG_DIR / 'ml_feature_stage_h24_val_mae.png'
FIG_ML_H24_HIGH_POLLUTION_BIAS = FIG_DIR / 'ml_h24_high_pollution_bias.png'

# Target contract
TARGET_COL = 'PM2.5'
KEYS = ['station', 'timestamp']

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'OUT_DIR: {OUT_DIR}')
print(f'FIG_DIR: {FIG_DIR}')


PROJECT_ROOT: F:\DOCUMENTOS\CIENCIA DE DATOS\PROYECTOS\portfolio_data_science\air-quality-forecasting-environmental-signals
OUT_DIR: F:\DOCUMENTOS\CIENCIA DE DATOS\PROYECTOS\portfolio_data_science\air-quality-forecasting-environmental-signals\data\interim\beijing_air_quality\ml_first_models
FIG_DIR: F:\DOCUMENTOS\CIENCIA DE DATOS\PROYECTOS\portfolio_data_science\air-quality-forecasting-environmental-signals\reports\figures\ml_first_models


In [6]:
# File presence checks (fail fast)
required = [
    ASSEMBLED_PATH,
    CONTRACT_JSON_PATH, SPLIT_DEF_PATH, LABEL_H1_PATH, LABEL_H24_PATH,
    BASELINE_TO_BEAT_PATH, BASELINE_SCORE_OVERALL_PATH, BASELINE_SCORE_BY_STATION_PATH, BASELINE_SCORE_HIGH_PATH,
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required input artifacts:\n' + '\n'.join(missing))
len(required)


9

In [7]:
contract = json.loads(CONTRACT_JSON_PATH.read_text(encoding='utf-8'))
contract_preview = {
    'horizons_hours': contract.get('horizons_hours'),
    'split': {
        'purge_gap_hours': contract.get('split', {}).get('purge_gap_hours'),
        'split_boundaries': contract.get('split', {}).get('split_boundaries'),
    },
    'high_pollution': contract.get('high_pollution'),
}
contract_preview


{'horizons_hours': {'primary': 24, 'secondary': 1, 'all': [1, 24]},
 'split': {'purge_gap_hours': 24,
  'split_boundaries': {'train_start': '2013-03-01 00:00:00',
   'train_end': '2015-12-18 15:00:00',
   'train_purge_start': '2015-12-18 16:00:00',
   'train_purge_end': '2015-12-19 15:00:00',
   'val_start': '2015-12-19 16:00:00',
   'val_end': '2016-07-24 19:00:00',
   'val_purge_start': '2016-07-24 20:00:00',
   'val_purge_end': '2016-07-25 19:00:00',
   'test_start': '2016-07-25 20:00:00',
   'test_end': '2017-02-28 23:00:00'}},
 'high_pollution': {'definition': 'train_only_p95_of_target_y',
  'global_p95_value_h1': 239.0,
  'global_p95_value_h24': 239.0}}

In [8]:
baseline_to_beat = pd.read_parquet(BASELINE_TO_BEAT_PATH)
baseline_to_beat.head(12)


,baseline,horizon_hours,split,slice,n,mae,rmse,r2,median_ae,bias,source
0,persistence,1,test,all,61106,10.737898,21.422994,0.946918,5.0,-0.015023,overall
1,persistence,1,val,all,60964,10.111968,20.132479,0.939539,5.0,-0.061184,overall
2,persistence,24,test,all,60242,65.373328,98.510997,-0.116852,43.0,0.442183,overall
3,persistence,24,val,all,60170,53.996261,89.424391,-0.211841,32.0,0.505185,overall
4,persistence,1,test,high_global_p95,4239,27.208304,43.863532,0.769629,17.0,-6.186365,high_pollution
5,persistence,1,test,high_station_p95,4202,27.194193,43.818402,0.775092,17.0,-6.051404,high_pollution
6,persistence,1,val,high_global_p95,2813,30.291859,55.376214,0.704576,17.0,-6.898329,high_pollution
7,persistence,1,val,high_station_p95,2811,30.206688,55.270246,0.709283,17.0,-6.963714,high_pollution
8,persistence,24,test,high_global_p95,4208,146.692253,183.192125,-3.000856,128.0,-116.585789,high_pollution
9,persistence,24,test,high_station_p95,4172,146.739693,183.249708,-2.916305,128.5,-117.639981,high_pollution


In [9]:
label_h24 = pd.read_parquet(LABEL_H24_PATH)
label_h1 = pd.read_parquet(LABEL_H1_PATH)

label_counts = (
    pd.concat([label_h1.assign(horizon_hours=1), label_h24.assign(horizon_hours=24)], ignore_index=True)
    .groupby(['horizon_hours', 'split'], as_index=False)
    .agg(n_rows=('y', 'size'), y_missing_rate=('y', lambda s: float(pd.Series(s).isna().mean())))
)
label_counts


,horizon_hours,split,n_rows,y_missing_rate
0,1,test,61574,0.0
1,1,train,288357,0.0
2,1,val,61526,0.0
3,24,test,61331,0.0
4,24,train,288081,0.0
5,24,val,61250,0.0


In [10]:
# Assembled history preview (read a narrow subset; full feature table is built later).
assembled_preview = pd.read_parquet(ASSEMBLED_PATH, columns=['station', 'timestamp'])
assembled_n_rows = int(assembled_preview.shape[0])
assembled_station_count = int(assembled_preview['station'].nunique())
{'assembled_n_rows': assembled_n_rows, 'n_stations': assembled_station_count}


{'assembled_n_rows': 420768, 'n_stations': 12}

## 3. Define supervised modeling rows and join keys

Analytical questions:
- What is one training row?
- What is `t_pred`? What is `t_target`?
- Why are labels reused from Notebook 03 instead of rebuilt here?

Why this matters for the project:
- The frozen contract is the backbone of comparability. If we rebuild labels here, we risk silent drift (thresholds, splits, purge gaps, or inclusion rules).

Row contract (one example):
- Key: `(station, t_pred)`
- Target: `y = PM2.5(t_target)` where `t_target = t_pred + horizon_hours`
- Split: assigned by `t_pred` under the Notebook 03 split definition


In [11]:
def standardize_label_index(li: pd.DataFrame, horizon_hours: int, contract: dict) -> pd.DataFrame:
    out = li.copy()

    required = ['station', 't_pred', 't_target', 'split', 'y']
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f'Label index missing required columns: {missing}')

    out['horizon_hours'] = int(horizon_hours)
    out['t_pred'] = pd.to_datetime(out['t_pred'])
    out['t_target'] = pd.to_datetime(out['t_target'])
    out['y'] = out['y'].astype(float)

    # Slice flags (prefer precomputed NB03 flags; fallback to contract global p95).
    if 'is_high_global_p95' in out.columns:
        out['high_global_p95'] = out['is_high_global_p95'].astype(bool)
    else:
        hp = contract.get('high_pollution', {})
        global_p95 = float(hp.get(f'global_p95_value_h{horizon_hours}', np.nan))
        if not np.isfinite(global_p95):
            out['high_global_p95'] = False
        else:
            out['high_global_p95'] = out['y'].astype(float) >= global_p95

    if 'is_high_station_p95' in out.columns:
        out['high_station_p95'] = out['is_high_station_p95'].astype(bool)
    else:
        out['high_station_p95'] = pd.NA

    keep = ['station', 't_pred', 't_target', 'split', 'y', 'horizon_hours', 'high_global_p95', 'high_station_p95']
    return out[keep].copy()


In [12]:
rows_h24 = standardize_label_index(label_h24, horizon_hours=24, contract=contract)
rows_h1 = standardize_label_index(label_h1, horizon_hours=1, contract=contract)

rows_h24.head(5)


,station,t_pred,t_target,split,y,horizon_hours,high_global_p95,high_station_p95
0,Aotizhongxin,2013-03-01 00:00:00,2013-03-02 00:00:00,train,22.0,24,False,False
1,Aotizhongxin,2013-03-01 01:00:00,2013-03-02 01:00:00,train,14.0,24,False,False
2,Aotizhongxin,2013-03-01 02:00:00,2013-03-02 02:00:00,train,13.0,24,False,False
3,Aotizhongxin,2013-03-01 03:00:00,2013-03-02 03:00:00,train,3.0,24,False,False
4,Aotizhongxin,2013-03-01 04:00:00,2013-03-02 04:00:00,train,3.0,24,False,False


## 4. Feature availability rules and leakage boundaries

Analytical question: are features strictly available at decision time `t_pred` without using the forecast window?

Why this matters for the project: leakage would inflate ML performance and break comparability with Notebook 04 baselines.

Rule table (compact):

| Category | Allowed | Forbidden |
|---|---|---|
| Identity / calendar | station; hour/dayofweek/month at `t_pred` | any calendar derived from `t_target` |
| Observed history | any observed variable timestamped `≤ t_pred` | any value in `(t_pred, t_target]` |
| Rolling windows | rolling summaries using history ending at `t_pred` | windows that extend into `(t_pred, t_target]` |
| Preprocessing | fit on train, apply to val/test | fit on val/test or combined splits |


## 5. Build timestamp-level feature table

Analytical questions:
- What features can be computed at the station-hour level using only history up to each timestamp?
- What is the coverage/missingness of these features before we join to label rows?

Why this matters for the project:
- Building features at `(station, timestamp)` keeps feature computation separate from label construction.
- No imputation is performed; missingness affects retained-row counts and is part of the model evidence.


In [13]:
# Minimal column subset (expected in assembled data); keep only what we need for F0–F3.
needed_cols = ['station', 'timestamp', TARGET_COL, 'TEMP', 'PRES', 'DEWP', 'WSPM', 'RAIN']
assembled_full = pd.read_parquet(ASSEMBLED_PATH)
available_cols = set(assembled_full.columns)
use_cols = [c for c in needed_cols if c in available_cols]
df_hist = assembled_full[use_cols].copy()
del assembled_full

df_hist['timestamp'] = pd.to_datetime(df_hist['timestamp'])
df_hist = df_hist.sort_values(['station', 'timestamp']).reset_index(drop=True)

df_hist.head(3)


,station,timestamp,PM2.5,TEMP,PRES,DEWP,WSPM,RAIN
0,Aotizhongxin,2013-03-01 00:00:00,4.0,-0.7,1023.0,-18.8,4.4,0.0
1,Aotizhongxin,2013-03-01 01:00:00,8.0,-1.1,1023.2,-18.2,4.7,0.0
2,Aotizhongxin,2013-03-01 02:00:00,7.0,-1.1,1023.5,-18.2,5.6,0.0


In [14]:
# Timestamp-level core observed values at t (rename with *_t suffix).
feat_ts = df_hist[['station', 'timestamp']].copy()
feat_ts['pm25_t'] = df_hist[TARGET_COL].astype(float)

if 'TEMP' in df_hist.columns:
    feat_ts['TEMP_t'] = df_hist['TEMP'].astype(float)
if 'PRES' in df_hist.columns:
    feat_ts['PRES_t'] = df_hist['PRES'].astype(float)
if 'DEWP' in df_hist.columns:
    feat_ts['DEWP_t'] = df_hist['DEWP'].astype(float)
if 'WSPM' in df_hist.columns:
    feat_ts['WSPM_t'] = df_hist['WSPM'].astype(float)
if 'RAIN' in df_hist.columns:
    feat_ts['RAIN_t'] = df_hist['RAIN'].astype(float)

feat_ts.shape


(420768, 8)

In [15]:
# Rolling summaries up to t_pred (F2).
MIN_OBS_ROLL24 = 18

pm25 = df_hist[TARGET_COL].astype(float)
g = df_hist.groupby('station', sort=False)

feat_ts['pm25_roll24_mean'] = g[TARGET_COL].transform(lambda s: s.astype(float).rolling(window=24, min_periods=MIN_OBS_ROLL24).mean())
feat_ts['pm25_roll24_median'] = g[TARGET_COL].transform(lambda s: s.astype(float).rolling(window=24, min_periods=MIN_OBS_ROLL24).median())
feat_ts['pm25_roll24_std'] = g[TARGET_COL].transform(lambda s: s.astype(float).rolling(window=24, min_periods=MIN_OBS_ROLL24).std())
feat_ts['pm25_roll24_min'] = g[TARGET_COL].transform(lambda s: s.astype(float).rolling(window=24, min_periods=MIN_OBS_ROLL24).min())
feat_ts['pm25_roll24_max'] = g[TARGET_COL].transform(lambda s: s.astype(float).rolling(window=24, min_periods=MIN_OBS_ROLL24).max())

if 'TEMP' in df_hist.columns:
    feat_ts['TEMP_roll24_mean'] = g['TEMP'].transform(lambda s: s.astype(float).rolling(window=24, min_periods=MIN_OBS_ROLL24).mean())
    feat_ts['TEMP_roll24_std'] = g['TEMP'].transform(lambda s: s.astype(float).rolling(window=24, min_periods=MIN_OBS_ROLL24).std())
if 'WSPM' in df_hist.columns:
    feat_ts['WSPM_roll24_mean'] = g['WSPM'].transform(lambda s: s.astype(float).rolling(window=24, min_periods=MIN_OBS_ROLL24).mean())
    feat_ts['WSPM_roll24_std'] = g['WSPM'].transform(lambda s: s.astype(float).rolling(window=24, min_periods=MIN_OBS_ROLL24).std())

feat_ts[['pm25_roll24_mean', 'pm25_roll24_median']].head(3)


,pm25_roll24_mean,pm25_roll24_median
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN


In [16]:
# Bounded lag features (F3).
feat_ts['pm25_lag_1'] = g[TARGET_COL].shift(1).astype(float)
feat_ts['pm25_lag_3'] = g[TARGET_COL].shift(3).astype(float)
feat_ts['pm25_lag_6'] = g[TARGET_COL].shift(6).astype(float)
feat_ts['pm25_lag_12'] = g[TARGET_COL].shift(12).astype(float)
feat_ts['pm25_lag_24'] = g[TARGET_COL].shift(24).astype(float)

if 'TEMP' in df_hist.columns:
    feat_ts['TEMP_lag_1'] = g['TEMP'].shift(1).astype(float)
    feat_ts['TEMP_lag_24'] = g['TEMP'].shift(24).astype(float)
if 'WSPM' in df_hist.columns:
    feat_ts['WSPM_lag_1'] = g['WSPM'].shift(1).astype(float)
    feat_ts['WSPM_lag_24'] = g['WSPM'].shift(24).astype(float)

feat_ts[['pm25_lag_1', 'pm25_lag_24']].head(3)


,pm25_lag_1,pm25_lag_24
0,NaN,NaN
1,4.0,NaN
2,8.0,NaN


In [17]:
# Timestamp-level missingness snapshot (small, to support later retained-row discussion).
miss = (
    feat_ts.drop(columns=['station', 'timestamp'])
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(12)
)
miss


pm25_lag_24           0.021427
pm25_lag_12           0.021107
pm25_lag_6            0.020940
pm25_lag_3            0.020855
pm25_lag_1            0.020798
pm25_t                0.020769
pm25_roll24_mean      0.018616
pm25_roll24_median    0.018616
pm25_roll24_max       0.018616
pm25_roll24_std       0.018616
pm25_roll24_min       0.018616
TEMP_lag_24           0.001630
dtype: float64

## 6. Define staged feature sets F0–F3

Analytical question: which feature families add measurable value (and at what cost) beyond temporal baselines?

Why this matters for the project: feature stages provide a controlled expansion of complexity so we can attribute gains to specific feature families (calendar, weather context, rolling summaries, explicit lags).

Required output: a feature-stage table (feature_set, families, hypothesis, expected value, risk/cost).


In [18]:
FEATURE_SETS = {
    'F0': {
        'features': ['pm25_t'],
        'notes': 'calendar + station + PM2.5 at t_pred',
    },
    'F1': {
        'features': ['pm25_t', 'TEMP_t', 'PRES_t', 'DEWP_t', 'WSPM_t', 'RAIN_t'],
        'notes': 'F0 + meteorology at t_pred',
    },
    'F2': {
        'features': [
            'pm25_t', 'TEMP_t', 'PRES_t', 'DEWP_t', 'WSPM_t', 'RAIN_t',
            'pm25_roll24_mean', 'pm25_roll24_median', 'pm25_roll24_std', 'pm25_roll24_min', 'pm25_roll24_max',
            'TEMP_roll24_mean', 'TEMP_roll24_std',
            'WSPM_roll24_mean', 'WSPM_roll24_std',
        ],
        'notes': 'F1 + rolling summaries up to t_pred',
    },
    'F3': {
        'features': [
            'pm25_t', 'TEMP_t', 'PRES_t', 'DEWP_t', 'WSPM_t', 'RAIN_t',
            'pm25_roll24_mean', 'pm25_roll24_median', 'pm25_roll24_std', 'pm25_roll24_min', 'pm25_roll24_max',
            'TEMP_roll24_mean', 'TEMP_roll24_std',
            'WSPM_roll24_mean', 'WSPM_roll24_std',
            'pm25_lag_1', 'pm25_lag_3', 'pm25_lag_6', 'pm25_lag_12', 'pm25_lag_24',
            'TEMP_lag_1', 'TEMP_lag_24',
            'WSPM_lag_1', 'WSPM_lag_24',
        ],
        'notes': 'F2 + bounded lags',
    },
}

# Remove features not present in feat_ts (e.g., if a weather column is absent in assembled data).
present = set(feat_ts.columns)
for k in FEATURE_SETS:
    FEATURE_SETS[k]['features'] = [f for f in FEATURE_SETS[k]['features'] if f in present]

feature_stage_table = pd.DataFrame([
    {
        'feature_set': fs,
        'included_feature_families': FEATURE_SETS[fs]['notes'],
        'hypothesis_tested': {
            'F0': 'Minimal supervised model adds value beyond persistence when given station+calendar context',
            'F1': 'Observed meteorology at t_pred adds predictive signal beyond F0',
            'F2': 'Recent level/volatility summaries up to t_pred improve forecasts beyond F1',
            'F3': 'Explicit bounded lags improve beyond rolling summaries (temporal structure)',
        }[fs],
        'expected_value': {
            'F0': 'Small or no gain vs persistence on h24; sanity check baseline ML behavior',
            'F1': 'Potential gain on regime changes driven by weather context',
            'F2': 'More stable gains if recent dynamics matter (level/volatility)',
            'F3': 'Gains if lag structure is not captured by simple rollups',
        }[fs],
        'risk/cost': {
            'F0': 'Low cost; minimal missingness',
            'F1': 'Missingness if weather values absent; more preprocessing',
            'F2': 'Retained rows drop early in series due to rolling windows',
            'F3': 'Further retained-row loss due to lag requirements; more features',
        }[fs],
    }
    for fs in ['F0', 'F1', 'F2', 'F3']
])
feature_stage_table


,feature_set,included_feature_families,hypothesis_tested,expected_value,risk/cost
0,F0,calendar + station + PM2.5 at t_pred,Minimal supervised model adds value beyond per...,Small or no gain vs persistence on h24; sanity...,Low cost; minimal missingness
1,F1,F0 + meteorology at t_pred,Observed meteorology at t_pred adds predictive...,Potential gain on regime changes driven by wea...,Missingness if weather values absent; more pre...
2,F2,F1 + rolling summaries up to t_pred,Recent level/volatility summaries up to t_pred...,More stable gains if recent dynamics matter (l...,Retained rows drop early in series due to roll...
3,F3,F2 + bounded lags,Explicit bounded lags improve beyond rolling s...,Gains if lag structure is not captured by simp...,Further retained-row loss due to lag requireme...


## 7. Build model matrices and retained-row reports

Analytical questions:
- After enforcing feature availability (no imputation), how many rows remain per split for each feature stage?
- Which features drive most missingness (and therefore denominator differences)?

Why this matters for the project:
- Scores are only comparable when denominator differences are understood. Availability is part of the model evidence, not a side detail.

Required visible outputs:
- retained-n table by horizon × feature_set × split
- top missing features by feature_set (if relevant)


In [19]:
def add_calendar_features(rows: pd.DataFrame) -> pd.DataFrame:
    out = rows.copy()
    t = pd.to_datetime(out['t_pred'])
    out['hour'] = t.dt.hour.astype('int16')
    out['dayofweek'] = t.dt.dayofweek.astype('int16')
    out['month'] = t.dt.month.astype('int16')
    return out


In [20]:
def build_design_matrix(rows: pd.DataFrame, feat_ts: pd.DataFrame, feature_set: str) -> pd.DataFrame:
    # Join label rows to timestamp-level features by (station, t_pred == timestamp).
    base = add_calendar_features(rows)
    joined = base.merge(
        feat_ts.rename(columns={'timestamp': 't_pred'}),
        on=['station', 't_pred'],
        how='left',
    )

    required_feats = FEATURE_SETS[feature_set]['features']
    keep = [
        'station', 't_pred', 't_target', 'split', 'y', 'horizon_hours',
        'high_global_p95', 'high_station_p95',
        'hour', 'dayofweek', 'month',
        *required_feats,
    ]
    keep = [c for c in keep if c in joined.columns]
    out = joined[keep].copy()

    # No label rebuilding here; but we still drop any rows with missing y (cannot train/score).
    out = out.loc[out['y'].notna()].copy()

    # Drop rows missing any required feature for this feature_set (no imputation).
    missing_mask = out[required_feats].isna().any(axis=1) if required_feats else pd.Series(False, index=out.index)
    out = out.loc[~missing_mask].copy()

    out['feature_set'] = feature_set
    return out


In [21]:
def retained_n_report(rows: pd.DataFrame, feature_set: str, horizon_hours: int) -> pd.DataFrame:
    total = (
        rows.groupby('split', as_index=False)
        .agg(n_total_rows=('y', 'size'))
    )

    dm = build_design_matrix(rows, feat_ts=feat_ts, feature_set=feature_set)
    kept = (
        dm.groupby('split', as_index=False)
        .agg(n_retained_rows=('y', 'size'))
    )

    out = total.merge(kept, on='split', how='left')
    out['n_retained_rows'] = out['n_retained_rows'].fillna(0).astype(int)
    out['retained_rate'] = out['n_retained_rows'] / out['n_total_rows']
    out.insert(0, 'feature_set', feature_set)
    out.insert(0, 'horizon_hours', int(horizon_hours))
    return out


In [22]:
retained_rows = []
for fs in ['F0', 'F1', 'F2', 'F3']:
    retained_rows.append(retained_n_report(rows_h1, feature_set=fs, horizon_hours=1))
    retained_rows.append(retained_n_report(rows_h24, feature_set=fs, horizon_hours=24))
retained_table = pd.concat(retained_rows, ignore_index=True)
retained_table


,horizon_hours,feature_set,split,n_total_rows,n_retained_rows,retained_rate
0,1,F0,test,61574,61106,0.992399
1,1,F0,train,288357,286558,0.993761
2,1,F0,val,61526,60964,0.990866
3,24,F0,test,61331,60242,0.982244
4,24,F0,train,288081,283641,0.984588
5,24,F0,val,61250,60170,0.982367
6,1,F1,test,61574,60887,0.988843
7,1,F1,train,288357,286372,0.993116
8,1,F1,val,61526,60964,0.990866
9,24,F1,test,61331,60022,0.978657


In [23]:
def top_missing_features(rows: pd.DataFrame, feature_set: str, top_k: int = 10) -> pd.Series:
    base = add_calendar_features(rows)
    joined = base.merge(
        feat_ts.rename(columns={'timestamp': 't_pred'}),
        on=['station', 't_pred'],
        how='left',
    )
    feats = FEATURE_SETS[feature_set]['features']
    feats = [f for f in feats if f in joined.columns]
    if not feats:
        return pd.Series(dtype=float)
    return joined[feats].isna().mean().sort_values(ascending=False).head(top_k)

top_missing_h24_F3 = top_missing_features(rows_h24, feature_set='F3', top_k=12)
top_missing_h24_F3


pm25_lag_24           0.018434
pm25_lag_12           0.017479
pm25_lag_6            0.016766
pm25_lag_1            0.016500
pm25_lag_3            0.016495
pm25_t                0.016094
pm25_roll24_mean      0.014586
pm25_roll24_median    0.014586
pm25_roll24_max       0.014586
pm25_roll24_std       0.014586
pm25_roll24_min       0.014586
TEMP_lag_24           0.001666
dtype: float64

## 8. Candidate model slate and preprocessing rules

Analytical question: do linear vs nonlinear candidates respond differently to feature stages, while remaining leakage-safe?

Why this matters for the project: we want a bounded candidate slate that is interpretable enough to debug if ML fails to beat persistence, without model hopping.

Required output: candidate table (model, role, preprocessing, grid, why included, cost/risk).


In [24]:
# Candidate slate + grids (fixed; small, bounded).
ENABLE_RANDOM_FOREST = False

CANDIDATES = [
    {
        'model': 'Ridge',
        'role': 'stable linear reference',
        'preprocessing': 'one-hot(cat) + scale(num); fit on train only',
        'grid': {'alpha': [1.0, 10.0, 100.0]},
        'why_included': 'baseline supervised reference; usually hard to beat for tabular with limited tuning',
        'cost/risk': 'fast; may underfit nonlinearities',
    },
    {
        'model': 'ElasticNet',
        'role': 'sparse / regularized linear alternative',
        'preprocessing': 'one-hot(cat) + scale(num); fit on train only',
        'grid': {'alpha': [0.1, 1.0], 'l1_ratio': [0.1, 0.5], 'max_iter': [5000]},
        'why_included': 'tests whether sparse structure helps vs Ridge',
        'cost/risk': 'fast; may be sensitive to scaling and correlated features',
    },
    {
        'model': 'HistGradientBoostingRegressor',
        'role': 'nonlinear tabular candidate',
        'preprocessing': 'one-hot(cat); no scaling needed; fit on train only',
        'grid': {'max_depth': [3, 6], 'learning_rate': [0.05], 'max_iter': [400], 'min_samples_leaf': [20]},
        'why_included': 'captures nonlinear effects and interactions without heavy tuning',
        'cost/risk': 'moderate runtime; may overfit if misconfigured (bounded grid)',
    },
    {
        'model': 'RandomForestRegressor',
        'role': 'optional runtime-heavy comparison only',
        'preprocessing': 'one-hot(cat); no scaling needed; fit on train only',
        'grid': {'enabled': [False], 'n_estimators': [300], 'max_depth': [None, 20], 'min_samples_leaf': [10]},
        'why_included': 'sanity check; often competitive but expensive',
        'cost/risk': 'runtime heavy; disabled by default',
    },
]
candidate_table = pd.DataFrame(CANDIDATES)
candidate_table


,model,role,preprocessing,grid,why_included,cost/risk
0,Ridge,stable linear reference,one-hot(cat) + scale(num); fit on train only,"{'alpha': [1.0, 10.0, 100.0]}",baseline supervised reference; usually hard to...,fast; may underfit nonlinearities
1,ElasticNet,sparse / regularized linear alternative,one-hot(cat) + scale(num); fit on train only,"{'alpha': [0.1, 1.0], 'l1_ratio': [0.1, 0.5], ...",tests whether sparse structure helps vs Ridge,fast; may be sensitive to scaling and correlat...
2,HistGradientBoostingRegressor,nonlinear tabular candidate,one-hot(cat); no scaling needed; fit on train ...,"{'max_depth': [3, 6], 'learning_rate': [0.05],...",captures nonlinear effects and interactions wi...,moderate runtime; may overfit if misconfigured...
3,RandomForestRegressor,optional runtime-heavy comparison only,one-hot(cat); no scaling needed; fit on train ...,"{'enabled': [False], 'n_estimators': [300], 'm...",sanity check; often competitive but expensive,runtime heavy; disabled by default


In [25]:
def build_preprocessor(cat_cols: list[str], num_cols: list[str], scale_numeric: bool) -> ColumnTransformer:
    cat = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32)
    transformers = []
    transformers.append(('cat', cat, cat_cols))
    if num_cols:
        if scale_numeric:
            transformers.append(('num', Pipeline([('scaler', StandardScaler())]), num_cols))
        else:
            transformers.append(('num', 'passthrough', num_cols))
    return ColumnTransformer(transformers=transformers, remainder='drop', verbose_feature_names_out=False)


## 9. Train/validation protocol and selection rule

Analytical question: which (model, feature_set) is the best candidate by **validation MAE on h24, slice=all**?

Why this matters for the project: selection must be honest (validation only). Test is reported after selection, not used to choose.

Protocol (fixed):
- Fit on **train only**
- Select by **validation MAE** on h24, slice=all
- Report test metrics after selection; no tuning based on test


In [26]:
def iter_param_grid(grid: dict) -> list[dict]:
    """Small fixed grids only (no search loops).

    Returns a list of param dicts. Uses itertools.product for readability.
    """
    keys = list(grid.keys())
    if not keys:
        return [{}]

    vals = [grid[k] for k in keys]
    out = []
    for combo in itertools.product(*vals):
        out.append({k: combo[i] for i, k in enumerate(keys)})
    return out


In [27]:
def make_model(model_name: str, params: dict) -> object:
    if model_name == 'Ridge':
        return Ridge(alpha=float(params['alpha']))
    if model_name == 'ElasticNet':
        return ElasticNet(alpha=float(params['alpha']), l1_ratio=float(params['l1_ratio']), max_iter=int(params.get('max_iter', 5000)))
    if model_name == 'HistGradientBoostingRegressor':
        return HistGradientBoostingRegressor(
            max_depth=int(params['max_depth']),
            learning_rate=float(params['learning_rate']),
            max_iter=int(params['max_iter']),
            min_samples_leaf=int(params['min_samples_leaf']),
            random_state=RANDOM_STATE,
        )
    if model_name == 'RandomForestRegressor':
        if not ENABLE_RANDOM_FOREST:
            raise ValueError('RandomForest disabled by default (ENABLE_RANDOM_FOREST=False)')
        return RandomForestRegressor(
            n_estimators=int(params.get('n_estimators', 300)),
            max_depth=params.get('max_depth', None),
            min_samples_leaf=int(params.get('min_samples_leaf', 10)),
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    raise ValueError(f'Unknown model: {model_name}')


## 10. Score ML candidates under the Notebook 04 metric contract

Analytical question: under the same metric contract as Notebook 04, how do ML candidates perform overall, by station, and on high-pollution slices?

Why this matters for the project: we need apples-to-apples comparison to baselines, including bias diagnostics in high-pollution slices.

Metrics (must match Notebook 04): MAE, RMSE, R², MedianAE, Bias.
Slices: all; high_global_p95; high_station_p95 (if available).
Denominator: per (model, feature_set) after dropping rows with missing required features (no imputation).


In [28]:
def score_one(y: pd.Series, yhat: pd.Series) -> dict:
    y = y.astype(float)
    yhat = yhat.astype(float)
    mask = y.notna() & yhat.notna()
    yy = y.loc[mask]
    yh = yhat.loc[mask]
    n = int(len(yy))
    if n == 0:
        return {'n': 0, 'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'median_ae': np.nan, 'bias': np.nan}

    err = yh - yy
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(np.square(err))))
    median_ae = float(np.median(np.abs(err)))
    bias = float(np.mean(err))
    sst = float(np.sum(np.square(yy - float(np.mean(yy)))))
    sse = float(np.sum(np.square(err)))
    r2 = float(1.0 - (sse / sst)) if sst > 0 else np.nan

    return {'n': n, 'mae': mae, 'rmse': rmse, 'r2': r2, 'median_ae': median_ae, 'bias': bias}


In [29]:
def score_overall(pred_df: pd.DataFrame, slice_name: str) -> dict:
    return score_one(pred_df['y'], pred_df['y_hat'])


In [30]:
def compute_scores(pred: pd.DataFrame, horizon_hours: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Overall scores by split/slice and by-station scores (slice=all only).
    rows_overall = []
    rows_high = []

    slice_defs = [('all', None)]
    if 'high_global_p95' in pred.columns:
        slice_defs.append(('high_global_p95', ('high_global_p95', True)))
    if 'high_station_p95' in pred.columns and pred['high_station_p95'].notna().any():
        slice_defs.append(('high_station_p95', ('high_station_p95', True)))

    for (model, feature_set, split), g in pred.groupby(['model', 'feature_set', 'split'], sort=True):
        for slice_name, cond in slice_defs:
            if cond is None:
                sub = g
            else:
                col, val = cond
                sub = g.loc[g[col] == val]
            m = score_one(sub['y'], sub['y_hat'])
            row = {
                'model': model,
                'feature_set': feature_set,
                'horizon_hours': int(horizon_hours),
                'split': split,
                'slice': slice_name,
                **m,
            }
            if slice_name == 'all':
                rows_overall.append(row)
            else:
                rows_high.append(row)

    overall = pd.DataFrame(rows_overall)
    high = pd.DataFrame(rows_high)

    by_station_rows = []
    for (model, feature_set, split, station), g in pred.groupby(['model', 'feature_set', 'split', 'station'], sort=True):
        m = score_one(g['y'], g['y_hat'])
        by_station_rows.append({
            'model': model,
            'feature_set': feature_set,
            'horizon_hours': int(horizon_hours),
            'station': station,
            'split': split,
            **m,
        })
    by_station = pd.DataFrame(by_station_rows)

    return overall, by_station, high


## 11. Compare ML candidates against temporal baselines

Analytical questions:
- Against `baseline_to_beat.parquet`, does any ML candidate improve MAE on h24 validation (slice=all)?
- On h24 high-pollution slices, does ML reduce the underprediction bias observed for persistence in Notebook 04?

Why this matters for the project:
- The baseline-to-beat defines the minimum bar. Improvements should be stated as deltas vs baseline, with denominators visible.

Required outputs in this section:
- `ml_vs_baseline` preview
- compact table for h24 validation all-slice
- compact table for h24 high-pollution slices
- baseline-to-beat vs best ML candidate table


In [31]:
# Build model matrices for each horizon × feature_set
design_matrices = {}
for horizon_hours, rows in [(1, rows_h1), (24, rows_h24)]:
    for fs in ['F0', 'F1', 'F2', 'F3']:
        dm = build_design_matrix(rows, feat_ts=feat_ts, feature_set=fs)
        design_matrices[(horizon_hours, fs)] = dm

# Blocker: no retained validation rows for h24 F0
n_h24_f0_val = int((design_matrices[(24, 'F0')]['split'] == 'val').sum())
if n_h24_f0_val == 0:
    raise ValueError('Blocker: no retained validation rows for h24 F0 after enforcing feature availability')

{k: v.shape for k, v in list(design_matrices.items())[:2]}


{(1, 'F0'): (408628, 13), (1, 'F1'): (408223, 18)}

In [32]:
# Availability report (persisted): retained rows + per-feature missingness (pre-drop).
availability_rows = []
missing_rows = []

for horizon_hours, rows in [(1, rows_h1), (24, rows_h24)]:
    rows_cal = add_calendar_features(rows)
    joined = rows_cal.merge(feat_ts.rename(columns={'timestamp': 't_pred'}), on=['station', 't_pred'], how='left')

    for fs in ['F0', 'F1', 'F2', 'F3']:
        feats = FEATURE_SETS[fs]['features']
        feats = [f for f in feats if f in joined.columns]

        for split, g in joined.groupby('split', sort=True):
            n_total = int(g.shape[0])
            if feats:
                retained_mask = ~g[feats].isna().any(axis=1)
                n_retained = int(retained_mask.sum())
                miss_rates = g[feats].isna().mean().sort_values(ascending=False)
                for feat_name, mr in miss_rates.head(20).items():
                    missing_rows.append({
                        'horizon_hours': int(horizon_hours),
                        'feature_set': fs,
                        'split': split,
                        'feature': feat_name,
                        'missing_rate': float(mr),
                    })
            else:
                n_retained = n_total

            availability_rows.append({
                'horizon_hours': int(horizon_hours),
                'feature_set': fs,
                'split': split,
                'n_total_rows': n_total,
                'n_retained_rows': n_retained,
                'retained_rate': float(n_retained / n_total) if n_total else np.nan,
            })

availability_report = pd.DataFrame(availability_rows)
availability_missing_top = pd.DataFrame(missing_rows)

availability_report.sort_values(['horizon_hours', 'feature_set', 'split']).head(24)


,horizon_hours,feature_set,split,n_total_rows,n_retained_rows,retained_rate
0,1,F0,test,61574,61106,0.992399
1,1,F0,train,288357,286558,0.993761
2,1,F0,val,61526,60964,0.990866
3,1,F1,test,61574,60887,0.988843
4,1,F1,train,288357,286372,0.993116
5,1,F1,val,61526,60964,0.990866
6,1,F2,test,61574,60173,0.977247
7,1,F2,train,288357,283908,0.984571
8,1,F2,val,61526,60602,0.984982
9,1,F3,test,61574,57628,0.935915


In [33]:
# Train/predict helper (fit on train only; predict train/val/test).
# Note: no imputation; dm already enforced feature availability per feature_set.
def fit_predict_one(dm: pd.DataFrame, model_name: str, params: dict) -> tuple[pd.DataFrame, Pipeline]:
    cat_cols = ['station', 'hour', 'dayofweek', 'month']
    num_cols = [c for c in dm.columns if c not in (
        'station', 't_pred', 't_target', 'split', 'y', 'horizon_hours', 'feature_set',
        'high_global_p95', 'high_station_p95',
        'hour', 'dayofweek', 'month',
    )]

    scale_numeric = model_name in ['Ridge', 'ElasticNet']
    pre = build_preprocessor(cat_cols=cat_cols, num_cols=num_cols, scale_numeric=scale_numeric)
    model = make_model(model_name, params=params)
    pipe = Pipeline([('pre', pre), ('model', model)])

    train = dm.loc[dm['split'] == 'train'].copy()
    if train.empty:
        raise ValueError('No training rows after feature availability filtering')

    pipe.fit(train[cat_cols + num_cols], train['y'].astype(float))

    out_parts = []
    for split in ['train', 'val', 'test']:
        sub = dm.loc[dm['split'] == split].copy()
        if sub.empty:
            continue
        y_hat = pipe.predict(sub[cat_cols + num_cols])
        part = sub[['station', 't_pred', 't_target', 'split', 'y', 'horizon_hours', 'feature_set', 'high_global_p95', 'high_station_p95']].copy()
        part['model'] = model_name
        part['params'] = json.dumps(params, sort_keys=True)
        part['y_hat'] = y_hat.astype(float)
        out_parts.append(part)

    out = pd.concat(out_parts, ignore_index=True)
    return out, pipe


In [34]:
# Train + predict all bounded candidates (RF disabled by default).
all_preds = []

for horizon_hours in [1, 24]:
    for fs in ['F0', 'F1', 'F2', 'F3']:
        dm = design_matrices[(horizon_hours, fs)]
        for cand in CANDIDATES:
            model_name = cand['model']
            if model_name == 'RandomForestRegressor' and not ENABLE_RANDOM_FOREST:
                continue

            grid = cand['grid'].copy()
            if model_name == 'RandomForestRegressor':
                grid = {k: v for k, v in grid.items() if k != 'enabled'}

            for params in iter_param_grid(grid):
                pred_df, _ = fit_predict_one(dm, model_name=model_name, params=params)
                pred_df['horizon_hours'] = int(horizon_hours)
                all_preds.append(pred_df)

ml_pred_all = pd.concat(all_preds, ignore_index=True)
ml_pred_all.shape


KeyboardInterrupt: 

In [ ]:
# Blocker: no valid model scores for h24 validation all-slice
if ml_pred_all.loc[(ml_pred_all['horizon_hours'] == 24) & (ml_pred_all['split'] == 'val')].empty:
    raise ValueError('Blocker: no predictions available for h24 validation')

pred_h1_all = ml_pred_all.loc[ml_pred_all['horizon_hours'] == 1].copy()
pred_h24_all = ml_pred_all.loc[ml_pred_all['horizon_hours'] == 24].copy()

# Per-horizon scoring
overall_h1, by_station_h1, high_h1 = compute_scores(pred_h1_all, horizon_hours=1)
overall_h24, by_station_h24, high_h24 = compute_scores(pred_h24_all, horizon_hours=24)

ml_scores_overall = pd.concat([overall_h1, overall_h24], ignore_index=True)
ml_scores_by_station = pd.concat([by_station_h1, by_station_h24], ignore_index=True)
ml_scores_high = pd.concat([high_h1, high_h24], ignore_index=True)

ml_scores_overall.head(10)


In [ ]:
# Candidate selection rule: best by validation MAE on h24, slice=all
val_h24 = ml_scores_overall.loc[(ml_scores_overall['horizon_hours'] == 24) & (ml_scores_overall['split'] == 'val') & (ml_scores_overall['slice'] == 'all')].copy()
val_h24 = val_h24.sort_values(['mae', 'rmse']).reset_index(drop=True)
best_row = val_h24.head(1)
if best_row.empty:
    raise ValueError('Blocker: no h24 validation all-slice scores available to select a candidate')

best_candidate = best_row[['model', 'feature_set']].iloc[0].to_dict()
best_candidate


In [ ]:
# ML vs baseline-to-beat comparison (deltas).
def select_best_ml(scores_all: pd.DataFrame, scores_high: pd.DataFrame) -> pd.DataFrame:
    def pick(df: pd.DataFrame, source: str) -> pd.DataFrame:
        s = df.copy()
        s = s.loc[s['n'] > 0].copy()
        s = s.sort_values(['horizon_hours', 'split', 'slice', 'mae', 'rmse']).reset_index(drop=True)
        winners = s.groupby(['horizon_hours', 'split', 'slice'], as_index=False).first()
        winners['source'] = source
        return winners

    best_overall = pick(scores_all, source='overall')
    best_high = pick(scores_high, source='high_pollution')
    out = pd.concat([best_overall, best_high], ignore_index=True)
    return out


In [ ]:
ml_best = select_best_ml(ml_scores_overall, ml_scores_high)

# Join against baseline_to_beat (reference floor)
join_keys = ['source', 'horizon_hours', 'split', 'slice']
ml_vs_baseline = ml_best.merge(baseline_to_beat.copy(), on=join_keys, how='left', suffixes=('_ml', '_baseline'))

# Deltas: ML - baseline (negative MAE delta is improvement).
ml_vs_baseline['mae_delta'] = ml_vs_baseline['mae_ml'] - ml_vs_baseline['mae_baseline']
ml_vs_baseline['mae_pct'] = ml_vs_baseline['mae_delta'] / ml_vs_baseline['mae_baseline']
ml_vs_baseline['bias_delta'] = ml_vs_baseline['bias_ml'] - ml_vs_baseline['bias_baseline']

ml_vs_baseline_out = (
    ml_vs_baseline[[
        'model_ml', 'feature_set_ml',
        'source', 'horizon_hours', 'split', 'slice',
        'baseline',
        'n_ml', 'mae_ml', 'rmse_ml', 'bias_ml',
        'n_baseline', 'mae_baseline', 'rmse_baseline', 'bias_baseline',
        'mae_delta', 'mae_pct', 'bias_delta',
    ]]
    .rename(columns={'model_ml': 'model', 'feature_set_ml': 'feature_set'})
)

ml_vs_baseline_out.head(12)


In [ ]:
# Required compact tables for h24 validation
h24_val_all = ml_vs_baseline_out.loc[(ml_vs_baseline_out['horizon_hours'] == 24) & (ml_vs_baseline_out['split'] == 'val') & (ml_vs_baseline_out['slice'] == 'all')].copy()
h24_val_all


In [ ]:
h24_val_high = ml_vs_baseline_out.loc[(ml_vs_baseline_out['horizon_hours'] == 24) & (ml_vs_baseline_out['split'] == 'val') & (ml_vs_baseline_out['slice'].isin(['high_global_p95', 'high_station_p95']))].copy()
h24_val_high


In [ ]:
# Curated figures (inline + saved).
# Figure 1: h24 val all-slice MAE delta vs baseline (best ML per source/slice).
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
sub = ml_vs_baseline_out.loc[(ml_vs_baseline_out['horizon_hours'] == 24) & (ml_vs_baseline_out['split'] == 'val') & (ml_vs_baseline_out['slice'] == 'all')].copy()
if sub.empty:
    ax.text(0.5, 0.5, 'No h24 val all-slice results', ha='center', va='center')
else:
    ax.bar(['best_ml'], sub['mae_delta'].to_numpy(), color='tab:blue', alpha=0.85)
    ax.axhline(0.0, color='black', linewidth=1)
    ax.set_title('h24 validation MAE delta vs baseline-to-beat (best ML)')
    ax.set_ylabel('MAE delta (ML − baseline)')
plt.show()
fig.savefig(FIG_ML_VS_BASELINE_H24_VAL_MAE, dpi=150)
plt.close(fig)
str(FIG_ML_VS_BASELINE_H24_VAL_MAE)


In [ ]:
# Figure 2: h24 validation MAE by feature stage for each model (slice=all).
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
plot_df = ml_scores_overall.loc[(ml_scores_overall['horizon_hours'] == 24) & (ml_scores_overall['split'] == 'val') & (ml_scores_overall['slice'] == 'all')].copy()
if plot_df.empty:
    ax.text(0.5, 0.5, 'No h24 val scores', ha='center', va='center')
else:
    for model, g in plot_df.groupby('model', sort=True):
        gg = g.sort_values('feature_set')
        ax.plot(gg['feature_set'].astype(str).to_list(), gg['mae'].to_numpy(), marker='o', label=model)
    ax.set_title('h24 validation MAE by feature stage (slice=all)')
    ax.set_xlabel('feature_set')
    ax.set_ylabel('MAE')
    ax.legend(loc='best')
plt.show()
fig.savefig(FIG_ML_FEATURE_STAGE_H24_VAL_MAE, dpi=150)
plt.close(fig)
str(FIG_ML_FEATURE_STAGE_H24_VAL_MAE)


In [ ]:
# Figure 3: h24 high-pollution bias comparison (validation).
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
sub = ml_vs_baseline_out.loc[(ml_vs_baseline_out['horizon_hours'] == 24) & (ml_vs_baseline_out['split'] == 'val') & (ml_vs_baseline_out['slice'].isin(['high_global_p95', 'high_station_p95']))].copy()
if sub.empty:
    ax.text(0.5, 0.5, 'No high-pollution slice results available', ha='center', va='center')
else:
    x = np.arange(sub.shape[0])
    width = 0.35
    ax.bar(x - width/2, sub['bias_baseline'].to_numpy(), width=width, label='baseline', color='tab:red', alpha=0.75)
    ax.bar(x + width/2, sub['bias_ml'].to_numpy(), width=width, label='best_ml', color='tab:blue', alpha=0.75)
    ax.axhline(0.0, color='black', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(sub['slice'].astype(str).to_list(), rotation=30, ha='right')
    ax.set_title('h24 validation bias on high-pollution slices (best ML vs baseline-to-beat)')
    ax.set_ylabel('Bias (y_hat − y)')
    ax.legend(loc='best')
plt.show()
fig.savefig(FIG_ML_H24_HIGH_POLLUTION_BIAS, dpi=150)
plt.close(fig)
str(FIG_ML_H24_HIGH_POLLUTION_BIAS)


## 12. Results summary: what improved, what did not, and what remains diagnostic

This section should be written **after** executing the results tables and curated figures.
Use cautious language: this notebook produces **candidate evidence**, not final project conclusions.

Required statements (evidence-backed):
- whether ML improved over persistence on validation h24 (slice=all)
- whether any improvement persists on test (after selection)
- whether high-pollution underprediction improved or worsened (bias)
- whether feature stages improved performance enough to justify complexity given retained rates
- what should be carried to the next diagnostic/finalist notebook


## 13. Persist ML artifacts and run summary

Analytical question: *(none; persistence/packaging)*

Why this matters: downstream notebooks should consume durable artifacts (predictions, scores, availability, ML-vs-baseline deltas) without recomputing this notebook.

Required output: artifact status table (path, exists, bytes).


In [ ]:
# Persist required artifacts
pred_h24_all.to_parquet(OUT_PRED_H24, index=False)
pred_h1_all.to_parquet(OUT_PRED_H1, index=False)

ml_scores_overall.to_parquet(OUT_SCORE_OVERALL, index=False)
ml_scores_by_station.to_parquet(OUT_SCORE_BY_STATION, index=False)
ml_scores_high.to_parquet(OUT_SCORE_HIGH, index=False)

availability_report.to_parquet(OUT_AVAILABILITY, index=False)
ml_vs_baseline_out.to_parquet(OUT_ML_VS_BASELINE, index=False)

run_summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'contract_dir': str(CONTRACT_DIR),
    'baseline_dir': str(BASELINE_DIR),
    'assembled_path': str(ASSEMBLED_PATH),
    'primary_horizon_hours': 24,
    'secondary_horizon_hours': 1,
    'feature_sets': {k: FEATURE_SETS[k]['features'] for k in FEATURE_SETS},
    'candidates': CANDIDATES,
    'enable_random_forest': bool(ENABLE_RANDOM_FOREST),
    'selection_rule': 'best by validation MAE on h24 slice=all',
    'selected_candidate': best_candidate,
    'artifacts': {
        'ml_predictions_h24': str(OUT_PRED_H24),
        'ml_predictions_h1': str(OUT_PRED_H1),
        'ml_scores_overall': str(OUT_SCORE_OVERALL),
        'ml_scores_by_station': str(OUT_SCORE_BY_STATION),
        'ml_scores_high_pollution': str(OUT_SCORE_HIGH),
        'availability_report': str(OUT_AVAILABILITY),
        'ml_vs_baseline': str(OUT_ML_VS_BASELINE),
        'fig_ml_vs_baseline_h24_val_mae': str(FIG_ML_VS_BASELINE_H24_VAL_MAE),
        'fig_ml_feature_stage_h24_val_mae': str(FIG_ML_FEATURE_STAGE_H24_VAL_MAE),
        'fig_ml_h24_high_pollution_bias': str(FIG_ML_H24_HIGH_POLLUTION_BIAS),
    },
}
OUT_RUN_SUMMARY.write_text(json.dumps(run_summary, indent=2), encoding='utf-8')

# Artifact status preview
artifact_paths = [
    OUT_PRED_H24, OUT_PRED_H1,
    OUT_SCORE_OVERALL, OUT_SCORE_BY_STATION, OUT_SCORE_HIGH,
    OUT_AVAILABILITY, OUT_ML_VS_BASELINE,
    FIG_ML_VS_BASELINE_H24_VAL_MAE, FIG_ML_FEATURE_STAGE_H24_VAL_MAE, FIG_ML_H24_HIGH_POLLUTION_BIAS,
    OUT_RUN_SUMMARY,
]
rows = []
for p in artifact_paths:
    rows.append({
        'artifact': p.name,
        'exists': bool(p.exists()),
        'bytes': int(p.stat().st_size) if p.exists() else None,
        'path': str(p),
    })
pd.DataFrame(rows)


## 14. Notebook close and handoff

Must state (after execution):
- what artifacts were produced
- what the next notebook should consume
- whether the next step should be diagnostics/finalist analysis or another bounded ML pass depending on evidence
